# Simple HDFS Prediction Translator

This notebook converts prediction-database rows into concise operator-facing explanations.

## 1. Import module

In [1]:
import os
import pandas as pd

import translate_hdfs as translator

## 2. Load event templates

In [2]:
TEMPLATE_PATH = r"Spell_result/out_templates.csv"

if os.path.exists(TEMPLATE_PATH):
    templates = translator.load_event_templates(TEMPLATE_PATH)
    print(f"Loaded {len(templates)} event templates.")
else:
    templates = {}
    print("Template file not found. Event labels will use EventID only.")

Loaded 31 event templates.


## 3. Build normal-training reference

In [3]:
TRAIN_PATH = r"Dataset/HDFS/hdfs_train"
BOS_ID = 32  # adjust if your model uses a different BOS token

if os.path.exists(TRAIN_PATH):
    training_sequences = translator.load_training_sequences_from_txt(TRAIN_PATH, bos_id=BOS_ID)
    transition_reference = translator.build_training_reference(training_sequences, bos_id=BOS_ID)
    print(f"Loaded {len(training_sequences)} training sequences.")
    print(f"Known EventIDs: {sorted(transition_reference.known_event_ids)}")
else:
    transition_reference = None
    print("Training file not found. Transition checks will be skipped.")

Loaded 4982 training sequences.
Known EventIDs: [1, 2, 3, 4, 5, 6, 7, 8, 9, 13, 23, 25]


## 4. Load prediction database

In [5]:
PREDICTION_DB_PATH = r"prediction_db.csv"

prediction_db = pd.read_csv(PREDICTION_DB_PATH)
prediction_db.head()

C:\Users\miqba\AppData\Local\Temp\ipykernel_5436\2881101090.py:3: DtypeWarning: Columns (0: results_matrix, 1: sequence_results_matrix) have mixed types. Specify dtype option on import or set low_memory=False.
  prediction_db = pd.read_csv(PREDICTION_DB_PATH)


,dataset,seq_id,sequence,seq_len,input_seq,target_seq,pred_events,pred_probs,target_event,target_event_is_unseen,target_in_topk_pred,target_in_pred_pos,target_in_topk_pred_prob,predicted_status,results,results_matrix,sequence_predicted_status,sequence_results_matrix
0,train,0,32 1 1 2 1 3 4 3 4 3 4 5 5 5 23 23 23 13 13 13,20,"[32, 1, 1, 2, 1, 3, 4, 3, 4, 3]",[4],"[5, 4, 23, 3, 13, 25, 9, 6, 24]","[0.8432120084762573, 0.15490305423736572, 0.00...",4,False,True,2.0,0.154903,normal,True,NaN,normal,NaN
1,train,0,32 1 1 2 1 3 4 3 4 3 4 5 5 5 23 23 23 13 13 13,20,"[32, 1, 1, 2, 1, 3, 4, 3, 4, 3]","[4, 5]","[5, 23, 4, 3, 25, 13, 6, 9, 24]","[0.9995607733726501, 0.00026684088516049087, 8...",5,False,True,1.0,0.999561,normal,True,NaN,normal,NaN
2,train,0,32 1 1 2 1 3 4 3 4 3 4 5 5 5 23 23 23 13 13 13,20,"[32, 1, 1, 2, 1, 3, 4, 3, 4, 3]","[4, 5, 5]","[5, 23, 4, 25, 13, 9, 6, 3, 2]","[0.9908952713012695, 0.006652186159044504, 0.0...",5,False,True,1.0,0.990895,normal,True,NaN,normal,NaN
3,train,0,32 1 1 2 1 3 4 3 4 3 4 5 5 5 23 23 23 13 13 13,20,"[32, 1, 1, 2, 1, 3, 4, 3, 4, 3]","[4, 5, 5, 5]","[5, 23, 13, 4, 9, 25, 6, 3, 2]","[0.5166937708854675, 0.4775638282299042, 0.001...",5,False,True,1.0,0.516694,normal,True,NaN,normal,NaN
4,train,0,32 1 1 2 1 3 4 3 4 3 4 5 5 5 23 23 23 13 13 13,20,"[32, 1, 1, 2, 1, 3, 4, 3, 4, 3]","[4, 5, 5, 5, 23]","[23, 9, 5, 13, 25, 6, 4, 24, 8]","[0.9991132616996765, 0.0005334100569598377, 0....",23,False,True,1.0,0.999113,normal,True,NaN,normal,NaN


## 5. Translate selected prediction rows

In [7]:
# Balanced sample for translator

target_datasets = ["normal_test", "abnormal_test"]

# Only use small metadata columns to find sample indices
sample_meta = prediction_db[["dataset", "results_matrix"]]

sample_meta = sample_meta[
    sample_meta["dataset"].isin(target_datasets)
]

sample_idx = (
    sample_meta
    .groupby("results_matrix", group_keys=False)
    .head(3)
    .index
)

# Keep only columns needed by the translator
needed_cols = [
    "dataset",
    "seq_id",
    "results_matrix",
    "input_seq",
    "target_seq",
    "target_event",
    "target_event_is_unseen",
    "target_in_topk_pred",
    "target_in_pred_pos",
    "target_in_topk_pred_prob",
    "pred_events",
    "pred_probs",
]

needed_cols = [col for col in needed_cols if col in prediction_db.columns]

sample_df = prediction_db.loc[sample_idx, needed_cols].copy()

sample_df.shape

(12, 12)

In [8]:
operator_report = translator.translate_dataframe(
    sample_df,
    transition_reference=transition_reference,
    templates=templates,
    top_k=9,
    bos_id=BOS_ID,
)

show_cols = [
    "sequence_id", "dataset", "result_type", "status", "severity",
    "previous_event_id", "actual_event_id", "hit_topk", "rank_in_topk",
    "target_probability_pct", "unseen_event", "transition_status",
    "evidence_1", "evidence_2", "evidence_3", "operator_action"
]

show_cols = [col for col in show_cols if col in operator_report.columns]

operator_report[show_cols]

,sequence_id,dataset,result_type,status,severity,previous_event_id,actual_event_id,hit_topk,rank_in_topk,target_probability_pct,unseen_event,transition_status,evidence_1,evidence_2,evidence_3,operator_action
0,0,normal_test,True Negative,Normal,Normal,3,4,True,1.0,99.0%,False,seen,Actual event was inside top-9 at rank 1.,Model probability for the actual event was 99.0%.,Actual EventID was seen in the normal training...,Continue normal monitoring.
1,0,normal_test,True Negative,Normal,Normal,4,3,True,1.0,100.0%,False,seen,Actual event was inside top-9 at rank 1.,Model probability for the actual event was 100...,Actual EventID was seen in the normal training...,Continue normal monitoring.
2,0,normal_test,True Negative,Normal,Normal,3,4,True,1.0,100.0%,False,seen,Actual event was inside top-9 at rank 1.,Model probability for the actual event was 100...,Actual EventID was seen in the normal training...,Continue normal monitoring.
3,2,normal_test,False Positive,Anomaly,Medium,4,6,False,NaN,N/A,False,rare,Actual event was not found in the model top-9 ...,Actual EventID was seen in the normal training...,Transition 4 -> 6 was rare in normal training ...,Review the log context around transition 4 -> ...
4,17,normal_test,False Positive,Anomaly,Medium,4,6,False,NaN,N/A,False,rare,Actual event was not found in the model top-9 ...,Actual EventID was seen in the normal training...,Transition 4 -> 6 was rare in normal training ...,Review the log context around transition 4 -> ...
5,18,normal_test,False Positive,Anomaly,Medium,4,6,False,NaN,N/A,False,rare,Actual event was not found in the model top-9 ...,Actual EventID was seen in the normal training...,Transition 4 -> 6 was rare in normal training ...,Review the log context around transition 4 -> ...
6,0,abnormal_test,False Negative,Normal,Normal,4,6,True,8.0,0.0%,False,rare,Actual event was inside top-9 at rank 8.,Model probability for the actual event was 0.0%.,Actual EventID was seen in the normal training...,Continue normal monitoring.
7,0,abnormal_test,False Negative,Normal,Normal,6,5,True,1.0,95.7%,False,rare,Actual event was inside top-9 at rank 1.,Model probability for the actual event was 95.7%.,Actual EventID was seen in the normal training...,Continue normal monitoring.
8,0,abnormal_test,False Negative,Normal,Normal,5,5,True,1.0,81.9%,False,seen,Actual event was inside top-9 at rank 1.,Model probability for the actual event was 81.9%.,Actual EventID was seen in the normal training...,Continue normal monitoring.
9,200,abnormal_test,True Positive,Anomaly,Medium,4,7,False,NaN,N/A,False,unseen,Actual event was not found in the model top-9 ...,Actual EventID was seen in the normal training...,Transition 4 -> 7 was not seen in normal train...,Review the log context around transition 4 -> ...


## 6. Print one operator message

In [10]:
example_message = translator.format_operator_message(operator_report.iloc[10])
print(example_message)

Sequence 200: ANOMALY (Medium). Event 1: Receiving block <*> src <*> dest <*> occurred after Event 7: <*> Starting thread to transfer block <*> to <*> <*>.

Evidence:
1. Actual event was not found in the model top-9 candidates.
2. Actual EventID was seen in the normal training reference.
3. Transition 7 -> 1 was seen in normal training (100.0%).

Suggested check: Review the log context around transition 7 -> 1; check whether this event order is expected for the current HDFS block operation.
